In [1]:
import rasterio
import numpy as np
import os
import tensorflow as tf
import keras 

In [2]:
# Folder where bands are stored
band_folder = "Browser_images"

# List the bands in the correct order (adjust based on LCZNet requirements)
band_order = ["B02.tiff", "B03.tiff", "B04.tiff", "B05.tiff",
              "B06.tiff", "B07.tiff", "B08.tiff", "B09.tiff",
              "B11.tiff", "B12.tiff", "B8A.tiff"]

# Create full file paths
band_paths = [os.path.join(band_folder, band) for band in band_order]

In [3]:
# Initialize an empty list to hold the band data
bands = []
for band_path in band_paths:
    with rasterio.open(band_path) as src:
        bands.append(src.read(1))  # Read the first band of each file

# Stack bands into a single numpy array [channels, height, width]
stacked_bands = np.stack(bands, axis=0).astype(np.float32)

# Normalize the data (assuming LCZNet expects normalized input)
stacked_bands /= 5000.0  # Adjust this normalization based on your data

In [4]:
# The model expects 10 bands; exclude the extra band (e.g., drop B08.tiff)
stacked_bands = stacked_bands[:10, :, :]  # Use only the first 10 bands

In [5]:
from skimage.util import view_as_blocks
tile_size = 64
stride = 8

# Calculate padding for 32x32 patches
pad_height = (tile_size - stacked_bands.shape[1] % tile_size) % tile_size
pad_width = (tile_size - stacked_bands.shape[2] % tile_size) % tile_size

stacked_bands_padded = np.pad(
    stacked_bands,
    pad_width=((0, 0), (0, pad_height), (0, pad_width)),  # No padding for channels
    mode="constant",
    constant_values=0
)

# New dimensions after padding
print("Original shape:", stacked_bands.shape)
print("Padded shape:", stacked_bands_padded.shape)


Original shape: (10, 769, 1402)
Padded shape: (10, 832, 1408)


In [6]:
from skimage.util import view_as_windows
# Divide the padded image into overlapping tiles
tiles = view_as_windows(
    stacked_bands_padded, window_shape=(10, tile_size, tile_size), step=stride
)

# Reshape tiles into correct batch format
tiles = tiles.reshape(-1, 10, tile_size, tile_size)
tiles = np.moveaxis(tiles, 1, -1)

In [ ]:
# Load and predict  each tile
model = keras.models.load_model("LCZNet.h5", compile=False)
predictions = []
for tile in tiles:
    tile = np.expand_dims(tile, axis=0)
    pred = model.predict(tile, verbose=0)
    predictions.append(np.argmax(pred.squeeze(), axis=-1)+1)

In [ ]:
print(predictions)
'''Outputs
[15, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 15, 14, 14, 14, 15, 15, 15, 15, 15, 14, 14, 14, 15, 15, 14, 14, 14, 14, 14, 14, 14, 15, 14, 15, 15, 15, 15, 15, 15, 15, 15, 14, 15, 14, 14, 14, 14, 15, 15, 14, 7, 14, 7, 7, 7, 15, 15, 15, 15, 15, 15, 7, 7, 15, 14, 14, 14, 14, 14, 7, 7, 7, 7, 7, 7, 7, 15, 15, 14, 15, 15, 15, 7, 7, 15, 15, 15, 14, 14, 14, 14, 14, 7, 7, 7, 7, 7, 7, 7, 15, 7, 15, 15, 15, 7, 7, 15, 14, 15, 15, 14, 15, 14, 14, 7, 7, 7, 6, 7, 7, 7, 14, 14, 14, 14, 14, 7, 15, 15, 15, 14, 14, 15, 14, 14, 7, 15, 7, 7, 7, 7, 15, 7, 15, 15, 14, 14, 15, 7, 7, 14, 15, 15, 14, 14, 14, 14, 15, 15, 7, 7, 14, 14, 7, 14, 14, 15, 15, 15, 15, 15, 14, 14, 14, 14, 14, 14, 7, 15, 7, 7, 7, 7, 14, 14, 15, 15, 14, 15, 14, 14, 15, 14, 15, 14, 14, 14, 14, 15, 14, 14, 15, 7, 7, 15, 14, 7, 7, 15, 14, 14, 14, 14, 15, 15, 15, 14, 14, 14, 15, 14, 7, 14, 14, 15, 14, 14, 7, 15, 15, 15, 14, 15, 14, 14, 14, 15, 15, 15, 15, 15, 15, 15, 14, 14, 14, 7, 14, 14, 7, 15, 15, 15, 14, 14, 14, 14, 14, 14, 14, 14, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16]
'''
print(len(predictions))

In [ ]:
# Reshape predictions into 2D
# Reshape predictions into a higher-resolution output
rows = (stacked_bands_padded.shape[1] - tile_size) // stride + 1
cols = (stacked_bands_padded.shape[2] - tile_size) // stride + 1
predictions_array = np.array(predictions).reshape(rows, cols)

# Expand each tile back to a full-size raster
lcz_map_padded = np.block([
    [predictions_array[i, j] * np.ones((stride, stride), dtype=np.uint8)
     for j in range(cols)] for i in range(rows)
])

# Remove padding to match original size
lcz_map = lcz_map_padded[:stacked_bands.shape[1], :stacked_bands.shape[2]]

In [ ]:
# Save the updated LCZ map as a GeoTIFF
output_file = "lcz_map_high_res8.tif"

with rasterio.open(band_paths[0]) as src:
    transform = src.transform
    crs = src.crs

with rasterio.open(
    output_file, "w",
    driver="GTiff",
    height=lcz_map.shape[0],
    width=lcz_map.shape[1],
    count=1,
    dtype=lcz_map.dtype,
    crs=crs,
    transform=transform
) as dst:
    dst.write(lcz_map, 1)

print(f"High-resolution LCZ map saved at {output_file}")
